# Gemma-Titans Hybrid: TPU v5e Online Learning Prototype

This notebook implements a hybrid architecture combining **Gemma-2B** with **Titans Neural Long-Term Memory (NLTM)**.

**Architecture Key Features:**
- **Parallel Memory:** Titans memory runs alongside standard Attention.
- **Learned Gating:** A scalar gate balances short-term context and long-term memory.
- **Modular Weights:** We only train and save the "Titans Delta" (~50MB), keeping the 5GB Gemma weights frozen.

## 1. Environment Setup (TPU v5e-1)
Run the cells below to install the JAX/Flax stack and the official Gemma framework.

In [1]:
# 1. Install specific versions of shared dependencies first
# !pip install "ml-dtypes==0.4.0" "protobuf==5.28.3"


# 2. Install JAX for TPU
# !pip install "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html


# 3. Install TensorFlow (Required for Kauldron/Gemma internal type checking on TPU)
# !pip install tensorflow==2.18.0 tensorflow-tpu==2.18.0 --find-links=https://storage.googleapis.com/libtpu-tf-releases/index.html

# 4. Install project dependencies with critical version locks (without kauldron==1.3.0)
!pip install typeguard==4.4.1 gemma==3.3.0 flax==0.12.5 optax==0.2.6
!pip install einops einx treescope jaxtyping sentencepiece datasets kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 94.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.9/188.9 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 139.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.5/180.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.3/492.3 kB 40.9 MB/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.5/139.5 kB 7.2 MB/s eta 0:00:00


In [1]:
# Clone the repository
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 100 (delta 52), reused 71 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 610.69 KiB | 16.96 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/Titans_jax


## 2. Initialization & Model Surgery
We initialize the hybrid structure and extract the initial "Titans Delta" weights.

In [2]:
import jax
import jax.numpy as jnp
import os
import gc # Add garbage collection
os.environ['KAULDRON_TYPECHECK'] = '0'
os.environ['KD_CHECK_TYPES'] = '0'

# Nuclear option: Monkey-patch the internal kauldron type-checker
# try:
#     import kauldron.typing.type_check as kt_check
#     kt_check._check_argument_types = lambda *args, **kwargs: None
#     kt_check._check_return_type = lambda func, retval, *args, **kwargs: retval
#     print("Successfully monkey-patched kauldron type-checker.")
# except Exception as e:
#     print(f"Note: Could not monkey-patch kauldron: {e}")

from gemma.gm.nn import _config, _modules
from gemma_titans import GemmaTitansTransformer
from model_loader import stitch_hybrid_model, load_titans_delta

# 1. Define Gemma-2B Config
config_2b = _config.TransformerConfig(
    num_embed=256000,
    embed_dim=2048,
    hidden_dim=16384,
    num_heads=8,
    head_dim=256,
    num_kv_heads=1,
    final_logit_softcap=30.0,
    use_post_attn_norm=False,
    use_post_ffw_norm=False,
    attention_types=[_modules.AttentionType.GLOBAL] * 18,
)

model = GemmaTitansTransformer(config=config_2b, dtype=jnp.bfloat16)

print("Initializing Hybrid Structure (bfloat16)...")
rng = jax.random.PRNGKey(0)
dummy_tokens = jnp.ones((1, 1), dtype=jnp.int32)
# JIT compile init to run on TPU and prevent CPU RAM OOM
init_fn = jax.jit(model.init)
params = init_fn(rng, tokens=dummy_tokens)['params']
gc.collect() # Force free RAM
print("Initialization complete. CPU RAM freed.")

import orbax.checkpoint as ocp
import kagglehub
from gemma.gm.nn import ckpts as gm_ckpts

# Extract only the Titans parameters (The Delta) from random init
def filter_titans(path, val):
    path_str = "/".join([str(p.name) if hasattr(p, 'name') else str(p) for p in path])
    if 'memory' in path_str or 'memory_gate' in path_str:
        return val
    return None

titans_delta = jax.tree_util.tree_map_with_path(filter_titans, params)

# Save initial random titans delta
print("Saving initial Titans Delta to ./titans_delta_init...")
checkpointer = ocp.StandardCheckpointer()
if os.path.exists("./titans_delta_init"):
    import shutil
    shutil.rmtree("./titans_delta_init")
checkpointer.save(os.path.abspath("./titans_delta_init"), titans_delta)

print("Downloading official Gemma-2B weights from Kaggle...")
# Note: Ensure you have authenticated with kagglehub or set KAGGLE_USERNAME and KAGGLE_KEY environment variables.
ckpt_path = kagglehub.model_download("google/gemma/Flax/2b")

print(f"Loading Gemma weights from {ckpt_path}...")
base_gemma_params = gm_ckpts.load_from_path(ckpt_path)

print("Stitching official Gemma weights with initialized Titans memory...")
params = stitch_hybrid_model(base_gemma_params, titans_delta)
print("Model loaded and stitched successfully!")
gc.collect()

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Successfully monkey-patched kauldron type-checker (args & returns).
Initializing Hybrid Structure (bfloat16)...
Initialization complete. CPU RAM freed.


ImportError: cannot import name 'ckpts' from 'gemma.gm.nn' (/usr/local/lib/python3.12/dist-packages/gemma/gm/nn/__init__.py)

## 3. Dataset Preparation
We use a subset of OpenAssistant for dialogue training.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("OpenAssistant/oasst1", split="train[:100]")
print(f"Loaded {len(dataset)} dialogue samples.")

## 4. Phase 1: Continued Pre-Training (CPT)
We freeze the Gemma weights and train only the `memory` and `memory_gate` parameters.

In [ ]:
import optax

# Define the mask: Only train Titans parameters
def is_trainable(path, val):
    path_str = "/".join([str(p.name) for p in path])
    return 'memory' in path_str or 'memory_gate' in path_str

mask = jax.tree_util.tree_map_with_path(is_trainable, params)

# Setup Optimizer with masking
tx = optax.chain(
    optax.masked(optax.adam(learning_rate=1e-4), mask)
)
opt_state = tx.init(params)

@jax.jit
def train_step(params, opt_state, tokens):
    def loss_fn(p):
        logits = model.apply({'params': p}, tokens=tokens).logits
        # Simple Cross-Entropy Loss implementation would go here
        return jnp.mean(logits**2) # Dummy loss for prototype demo

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = tx.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

print("Starting training step...")
params, opt_state, loss = train_step(params, opt_state, dummy_tokens)
print(f"Initial loss: {loss}")

## 5. Dialogue Inference with Dynamic Memory
This demonstrates how the model updates its internal state (`TitansLayerCache`) during a conversation.

In [ ]:
def dialogue_turn(params, cache, user_input_tokens):
    # 1. Forward pass returns updated cache (including updated Titans memory)
    output = model.apply({'params': params}, tokens=user_input_tokens, cache=cache)
    logits = output.logits
    new_cache = output.cache

    # 2. Logic to extract the response token would go here
    return logits, new_cache

# Initial state
cache = model.init_cache(batch_size=1, dtype=jnp.bfloat16, cache_length=512)

print("Simulation: User enters a turn...")
user_tokens = jnp.array([[1, 2, 3, 4]])
logits, cache = dialogue_turn(params, cache, user_tokens)
print("Memory state updated automatically in cache.")

## 6. Save/Load Titans Delta
We save ONLY the trained memory weights to keep the file size minimal.

In [ ]:
import orbax.checkpoint as ocp

def save_titans_only(params, path):
    # Filter to save only Titans parameters
    titans_delta = jax.tree_util.tree_map_with_path(
        lambda path, val: val if is_trainable(path, val) else None,
        params
    )

    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(os.path.abspath(path), titans_delta, force=True)
    print(f"Saved Titans Delta to {path}")

save_titans_only(params, "./titans_delta_init")